In [4]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# Locate the dataset folder (Kaggle mounts inputs under /kaggle/input)
candidates = [
    Path(root) for root, _, files in os.walk("/kaggle/input")
    if "cnn_image_link.parquet" in files
]
assert len(candidates) == 1, candidates
DATA_DIR = candidates[0]

patch_files = sorted(DATA_DIR.glob("patches_*.npy"))

print("Data folder:", DATA_DIR)
print("Patch files:", len(patch_files))
print("Other files:", sorted(p.name for p in DATA_DIR.iterdir() if p.suffix != ".npy"))
print()
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| count:", torch.cuda.device_count())

Data folder: /kaggle/input/datasets/danielalux/wildfire-risk-portugal-cnn
Patch files: 54
Other files: ['cnn_image_link.parquet', 'model_dataset_full.parquet', 'monthly_patch_index.parquet', 'uint8_scaling.json']

PyTorch: 2.11.0+cu128
GPU available: True
GPU: Tesla T4 | count: 2


# 06. CNN Training on Monthly Sentinel-2 Composites

**Question.** Does a convolutional neural network that sees the most recent monthly
Sentinel-2 composite add information about wildfire occurrence beyond weather and
historical context?

**Design.** Two networks with identical tabular inputs are compared:

- **Image + tabular**: a pre-trained ResNet-18 branch reads the previous month's
  composite (B4, B8, B11; 125 × 125 pixels at 40 m), and a small fully connected branch
  reads the weather and context features;
- **Tabular only**: the same network without the image branch.

Both receive the same weather features (16) and context features (climatology,
persistence, vegetation and land fraction) used by the best tabular model in
Notebook 04. The difference between them measures the contribution of the image.

**Splits.** Hyperparameters and the number of epochs are chosen by training on
2017–2021 and validating on 2022. Final models are trained on 2017–2022 and evaluated on
2023, as in Notebook 04. The test period (2024–2025) is not used here.

**Variability.** Each network is trained with three random seeds. A difference between
the two networks is only considered meaningful if it exceeds the variation across seeds.

In [5]:
import json
import psutil

TARGET = "fire_next_7_days"

WEATHER_FEATURES = [
    "consecutive_dry_days", "potential_evaporation_sum_7d",
    "precipitation_sum_14d", "precipitation_sum_30d", "precipitation_sum_7d",
    "relative_humidity_mean_7d", "soil_moisture_change_7d",
    "soil_moisture_last_day", "soil_moisture_mean_7d", "solar_radiation_sum_7d",
    "temperature_max_7d", "temperature_mean_7d", "vpd_mean_7d",
    "water_balance_7d", "wind_speed_max_daily_mean_7d", "wind_speed_mean_7d",
]

# climatology_rate is computed later (leave-one-year-out), so it is not in the file
STATIC_CONTEXT = ["fires_prev_1w", "fires_prev_4w", "vegetation_fraction", "land_fraction"]

model_data = pd.read_parquet(
    DATA_DIR / "model_dataset_full.parquet",
    columns=["cell_id", "reference_date", "iso_year", TARGET]
            + WEATHER_FEATURES + STATIC_CONTEXT,
)

image_link = pd.read_parquet(DATA_DIR / "cnn_image_link.parquet")

data = model_data.merge(
    image_link[["cell_id", "reference_date", "image_year", "image_month", "patch_index"]],
    on=["cell_id", "reference_date"],
    how="left",
    validate="one_to_one",
)

with open(DATA_DIR / "uint8_scaling.json") as f:
    uint8_scaling = json.load(f)

print("Rows:", len(data))
print("Positives:", int(data[TARGET].sum()))
print("Missing values:", int(data[WEATHER_FEATURES + STATIC_CONTEXT].isna().sum().sum()))
print("Rows without image link:", int(data["patch_index"].isna().sum()))
print()
print(data.groupby("iso_year")[TARGET].agg(["size", "sum"]))

assert len(data) == 875_238
assert int(data[TARGET].sum()) == 49_082
assert data["patch_index"].notna().all()

Rows: 875238
Positives: 49082
Missing values: 0
Rows without image link: 0

           size   sum
iso_year             
2017      99372  8987
2018      95550  5845
2019      95550  5959
2020      95550  5715
2021      99372  4475
2022      99372  5146
2023      99372  3940
2024      95550  3812
2025      95550  5203


In [6]:
ram = psutil.virtual_memory()
print(f"RAM total: {ram.total / 1e9:.1f} GB | available: {ram.available / 1e9:.1f} GB")

RAM total: 33.7 GB | available: 23.3 GB


In [8]:
import time

start = time.time()
patches = {}

for i, path in enumerate(patch_files, start=1):
    year, month = int(path.stem.split("_")[1]), int(path.stem.split("_")[2])
    patches[(year, month)] = np.load(path)
    print(f"{i}/{len(patch_files)} {path.name} — {time.time() - start:.0f} s", flush=True)

print(f"Loaded {len(patches)} months in {time.time() - start:.0f} s")
ram = psutil.virtual_memory()
print(f"RAM available after loading: {ram.available / 1e9:.1f} GB")

1/54 patches_2017_04.npy — 0 s
2/54 patches_2017_05.npy — 0 s
3/54 patches_2017_06.npy — 0 s
4/54 patches_2017_07.npy — 0 s
5/54 patches_2017_08.npy — 0 s
6/54 patches_2017_09.npy — 0 s
7/54 patches_2018_04.npy — 0 s
8/54 patches_2018_05.npy — 0 s
9/54 patches_2018_06.npy — 1 s
10/54 patches_2018_07.npy — 1 s
11/54 patches_2018_08.npy — 1 s
12/54 patches_2018_09.npy — 1 s
13/54 patches_2019_04.npy — 1 s
14/54 patches_2019_05.npy — 1 s
15/54 patches_2019_06.npy — 1 s
16/54 patches_2019_07.npy — 1 s
17/54 patches_2019_08.npy — 1 s
18/54 patches_2019_09.npy — 1 s
19/54 patches_2020_04.npy — 1 s
20/54 patches_2020_05.npy — 1 s
21/54 patches_2020_06.npy — 1 s
22/54 patches_2020_07.npy — 1 s
23/54 patches_2020_08.npy — 1 s
24/54 patches_2020_09.npy — 1 s
25/54 patches_2021_04.npy — 1 s
26/54 patches_2021_05.npy — 1 s
27/54 patches_2021_06.npy — 2 s
28/54 patches_2021_07.npy — 2 s
29/54 patches_2021_08.npy — 2 s
30/54 patches_2021_09.npy — 2 s
31/54 patches_2022_04.npy — 2 s
32/54 patches_202

## 1. Tabular inputs

The tabular branch receives the same information as the best tabular model in Notebook 04
(XGB Weather + Context): 16 weather features, `fires_prev_1w`, `fires_prev_4w`,
`vegetation_fraction`, `land_fraction` and `climatology_rate`.

`climatology_rate` is computed exactly as in Notebook 04: leave-one-year-out within the
training years, and from all training years for the validation year.

Unlike decision trees, neural networks are sensitive to the scale of their inputs.
Heavy-tailed counts and precipitation totals (`fires_prev_1w`, `fires_prev_4w`,
`precipitation_sum_7d/14d/30d`) are log-transformed (`log1p`). All features are then
standardised with the mean and standard deviation of the **training years only**, and
clipped to ±5 standard deviations to limit the influence of extreme values.

In [10]:
def add_climatology_feature(train_part, apply_parts):
    """
    Adds 'climatology_rate' (historical fire rate per cell × month).
    Training rows: leave-one-year-out. Apply rows: all training years.
    """
    train_part = train_part.copy()
    train_part["month"] = train_part["reference_date"].dt.month

    total = (
        train_part.groupby(["cell_id", "month"])[TARGET]
        .agg(total_sum="sum", total_count="size").reset_index()
    )
    by_year = (
        train_part.groupby(["cell_id", "month", "iso_year"])[TARGET]
        .agg(year_sum="sum", year_count="size").reset_index()
    )

    loyo = by_year.merge(total, on=["cell_id", "month"])
    loyo["climatology_rate"] = (
        (loyo["total_sum"] - loyo["year_sum"])
        / (loyo["total_count"] - loyo["year_count"])
    )

    train_part = train_part.merge(
        loyo[["cell_id", "month", "iso_year", "climatology_rate"]],
        on=["cell_id", "month", "iso_year"], how="left", validate="many_to_one",
    )

    total["climatology_rate"] = total["total_sum"] / total["total_count"]

    applied = []
    for part in apply_parts:
        part = part.copy()
        part["month"] = part["reference_date"].dt.month
        part = part.merge(
            total[["cell_id", "month", "climatology_rate"]],
            on=["cell_id", "month"], how="left", validate="many_to_one",
        )
        applied.append(part)

    return train_part, applied


TABULAR_FEATURES = WEATHER_FEATURES + STATIC_CONTEXT + ["climatology_rate"]
LOG_FEATURES = [
    "fires_prev_1w", "fires_prev_4w",
    "precipitation_sum_7d", "precipitation_sum_14d", "precipitation_sum_30d",
]


def prepare_split(train_years, val_years):
    """Builds standardised tabular arrays, labels and image keys for one split."""
    val_years = np.atleast_1d(val_years)
    train_part = data[data["iso_year"].isin(train_years)]
    val_part = data[data["iso_year"].isin(val_years)]
    train_part, (val_part,) = add_climatology_feature(train_part, [val_part])

    log_idx = [TABULAR_FEATURES.index(c) for c in LOG_FEATURES]

    def to_array(df):
        X = df[TABULAR_FEATURES].to_numpy(np.float32).copy()
        X[:, log_idx] = np.log1p(X[:, log_idx])
        return X

    X_train, X_val = to_array(train_part), to_array(val_part)

    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0) + 1e-6
    X_train = np.clip((X_train - mean) / std, -5, 5)
    X_val = np.clip((X_val - mean) / std, -5, 5)

    def pack(df, X):
        return {
            "X": X.astype(np.float32),
            "y": df[TARGET].to_numpy(np.float32),
            "image_year": df["image_year"].to_numpy(np.int16),
            "image_month": df["image_month"].to_numpy(np.int8),
            "patch_index": df["patch_index"].to_numpy(np.int32),
            "reference_date": df["reference_date"].to_numpy(),
            "cell_id": df["cell_id"].to_numpy(),
        }

    return pack(train_part, X_train), pack(val_part, X_val)


# Tuning phase: 2017–2021 → 2022 | Final phase: 2017–2022 → 2023
tune_train, tune_val = prepare_split(range(2017, 2022), 2022)
final_train, final_val = prepare_split(range(2017, 2023), 2023)

for name, split in [("tune_train", tune_train), ("tune_val", tune_val),
                    ("final_train", final_train), ("final_val", final_val)]:
    print(f"{name:12s} rows={len(split['y']):7,d}  positives={int(split['y'].sum()):6,d}  "
          f"rate={split['y'].mean():.4f}  NaN={int(np.isnan(split['X']).sum())}")

tune_train   rows=485,394  positives=30,981  rate=0.0638  NaN=0
tune_val     rows= 99,372  positives= 5,146  rate=0.0518  NaN=0
final_train  rows=584,766  positives=36,127  rate=0.0618  NaN=0
final_val    rows= 99,372  positives= 3,940  rate=0.0396  NaN=0


In [11]:
rng = np.random.default_rng(0)
train_months = [(y, m) for (y, m) in patches if 2017 <= y <= 2021]

sample = np.stack([
    patches[train_months[rng.integers(len(train_months))]][rng.integers(3822)]
    for _ in range(2000)
]).astype(np.float32) / 255.0

BAND_MEAN = sample.mean(axis=(0, 2, 3))
BAND_STD = sample.std(axis=(0, 2, 3))

print("Band mean:", BAND_MEAN.round(4))
print("Band std: ", BAND_STD.round(4))

Band mean: [0.2367 0.4494 0.4356]
Band std:  [0.151  0.1732 0.1953]


## 2. Data pipeline and models

**Dataset.** Each sample returns the standardised tabular vector, the label and, for the
image network, the 125 × 125 × 3 patch of the previous month's composite, scaled to [0, 1]
and normalised per band with statistics from the training years. During training, images
are randomly flipped and rotated by multiples of 90°, so the network does not depend on a
fixed orientation of the landscape.

### Negative subsampling during training

The training period contains about **584,000 weekly observations**, of which only
**≈36,000 (6.2%)** are positive. Negative subsampling was initially planned to keep
training within the free Kaggle GPU budget. A speed test showed that, with all patches
held in memory, an epoch with all observations takes only about 4 minutes on a Tesla T4,
so the budget is not a binding constraint.

The negative sampling rate r is therefore treated as a **hyperparameter**: r = 0.25 (all
positives plus a new random 25% of the negatives in each epoch) is compared with r = 1.0
(all observations) on the 2022 tuning split, and the better value is retained.

**Models.**

- *Tabular only*: a small fully connected network (tabular features → 128 → 64), followed
  by a classification head.
- *Image + tabular*: the same tabular branch plus a **ResNet-18 pre-trained on ImageNet**,
  whose 512-dimensional image representation is projected to 64 dimensions and
  concatenated with the tabular representation before the same head.

The pre-trained backbone is fine-tuned with a smaller learning rate than the new layers,
so that useful pre-trained filters are not destroyed in the first epochs. Training uses
binary cross-entropy, the AdamW optimiser and mixed precision on the GPU.

In [15]:
import time
import torch.nn as nn
import torchvision
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import average_precision_score

DEVICE = torch.device("cuda")
torch.backends.cudnn.benchmark = True

MEAN_T = torch.tensor(BAND_MEAN).view(3, 1, 1)
STD_T = torch.tensor(BAND_STD).view(3, 1, 1)
NEG_RATE = 0.25


class FireDataset(Dataset):
    def __init__(self, split, use_image, augment):
        self.split = split
        self.use_image = use_image
        self.augment = augment

    def __len__(self):
        return len(self.split["y"])

    def __getitem__(self, i):
        tab = torch.from_numpy(self.split["X"][i])
        y = torch.tensor(self.split["y"][i])

        if not self.use_image:
            return torch.empty(0), tab, y

        key = (int(self.split["image_year"][i]), int(self.split["image_month"][i]))
        img = torch.from_numpy(patches[key][self.split["patch_index"][i]]).float() / 255.0
        img = (img - MEAN_T) / STD_T

        if self.augment:
            if torch.rand(1) < 0.5:
                img = torch.flip(img, dims=[2])
            img = torch.rot90(img, k=int(torch.randint(0, 4, (1,))), dims=[1, 2])

        return img, tab, y


def epoch_indices(y, rng, neg_rate=NEG_RATE):
    """All positives + a fresh random sample of negatives, shuffled."""
    pos = np.flatnonzero(y == 1)
    neg = np.flatnonzero(y == 0)
    neg_sample = rng.choice(neg, size=int(len(neg) * neg_rate), replace=False)
    idx = np.concatenate([pos, neg_sample])
    rng.shuffle(idx)
    return idx


def make_train_loader(split, use_image, rng, neg_rate=NEG_RATE, batch_size=256):
    dataset = FireDataset(split, use_image, augment=True)
    subset = Subset(dataset, epoch_indices(split["y"], rng, neg_rate))
    return DataLoader(subset, batch_size=batch_size, shuffle=False,
                      num_workers=4, pin_memory=True, drop_last=True)


def make_eval_loader(split, use_image, batch_size=512):
    dataset = FireDataset(split, use_image, augment=False)
    return DataLoader(dataset, batch_size=batch_size, shuffle=False,
                      num_workers=4, pin_memory=True)

## 3. Training procedure

Each network is trained for up to 20 epochs. After every epoch, Average Precision is
computed on the full validation year, and training stops when it has not improved for
3 consecutive epochs (early stopping). The weights of the best epoch are kept.

Early stopping is used **only in the tuning phase** (validation on 2022). In the final
phase (2017–2022 → 2023), each network is trained for the number of epochs selected in
tuning, so that 2023 is never used to make a training decision.

In [13]:
from torchvision.models import ResNet18_Weights


class FireNet(nn.Module):
    def __init__(self, n_tabular, use_image):
        super().__init__()
        self.use_image = use_image

        self.tabular = nn.Sequential(
            nn.Linear(n_tabular, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.ReLU(),
        )

        head_input = 64
        if use_image:
            backbone = torchvision.models.resnet18(weights=ResNet18_Weights.DEFAULT)
            backbone.fc = nn.Identity()          # keep the 512-d representation
            self.backbone = backbone
            self.image_proj = nn.Sequential(nn.Linear(512, 64), nn.ReLU())
            head_input = 128

        self.head = nn.Sequential(
            nn.Dropout(0.2), nn.Linear(head_input, 32), nn.ReLU(), nn.Linear(32, 1),
        )

    def forward(self, img, tab):
        z = self.tabular(tab)
        if self.use_image:
            z = torch.cat([z, self.image_proj(self.backbone(img))], dim=1)
        return self.head(z).squeeze(1)


def make_optimizer(model, lr_new=1e-3, lr_backbone=1e-4, weight_decay=1e-4):
    backbone_params = list(model.backbone.parameters()) if model.use_image else []
    backbone_ids = {id(p) for p in backbone_params}
    new_params = [p for p in model.parameters() if id(p) not in backbone_ids]

    groups = [{"params": new_params, "lr": lr_new}]
    if backbone_params:
        groups.append({"params": backbone_params, "lr": lr_backbone})
    return torch.optim.AdamW(groups, weight_decay=weight_decay)

In [14]:
loss_fn = nn.BCEWithLogitsLoss()


def train_one_epoch(model, loader, optimizer, scaler, max_batches=None):
    model.train()
    total_loss, n_batches = 0.0, 0
    for img, tab, y in loader:
        img = img.to(DEVICE, non_blocking=True)
        tab = tab.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16):
            loss = loss_fn(model(img, tab), y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        n_batches += 1
        if max_batches and n_batches >= max_batches:
            break
    return total_loss / n_batches


@torch.no_grad()
def predict(model, loader, max_batches=None):
    model.eval()
    probs, n_batches = [], 0
    for img, tab, _ in loader:
        img = img.to(DEVICE, non_blocking=True)
        tab = tab.to(DEVICE, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16):
            probs.append(torch.sigmoid(model(img, tab)).float().cpu().numpy())
        n_batches += 1
        if max_batches and n_batches >= max_batches:
            break
    return np.concatenate(probs)

In [25]:
torch.manual_seed(0)
rng = np.random.default_rng(0)

model = FireNet(len(TABULAR_FEATURES), use_image=True).to(DEVICE)
optimizer = make_optimizer(model)
scaler = torch.amp.GradScaler("cuda")

train_loader = make_train_loader(tune_train, use_image=True, rng=rng)
val_loader = make_eval_loader(tune_val, use_image=True)

# warm-up (cuDNN autotuning, worker start-up)
train_one_epoch(model, train_loader, optimizer, scaler, max_batches=5)

start = time.time()
train_one_epoch(model, train_loader, optimizer, scaler, max_batches=50)
train_time = (time.time() - start) / 50

start = time.time()
predict(model, val_loader, max_batches=50)
eval_time = (time.time() - start) / 50

n_train_batches = len(train_loader)
n_val_batches = len(val_loader)

print(f"Train: {train_time:.3f} s/batch × {n_train_batches} batches "
      f"≈ {train_time * n_train_batches / 60:.1f} min per epoch")
print(f"Eval:  {eval_time:.3f} s/batch × {n_val_batches} batches "
      f"≈ {eval_time * n_val_batches / 60:.1f} min per validation pass")

Train: 0.131 s/batch × 564 batches ≈ 1.2 min per epoch
Eval:  0.123 s/batch × 195 batches ≈ 0.4 min per validation pass


In [16]:
def fit(train_split, val_split, use_image, seed, neg_rate, max_epochs=20, patience=3,
        fixed_epochs=None):
    """
    Trains one network. With fixed_epochs=None, uses early stopping on val AP and
    returns the best epoch's weights; with fixed_epochs=k, trains exactly k epochs.
    """
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)

    model = FireNet(len(TABULAR_FEATURES), use_image).to(DEVICE)
    optimizer = make_optimizer(model)
    scaler = torch.amp.GradScaler("cuda")
    val_loader = make_eval_loader(val_split, use_image)
    y_val = val_split["y"]

    n_epochs = fixed_epochs or max_epochs
    history, best_ap, best_epoch, best_state, bad_epochs = [], -1.0, 0, None, 0

    for epoch in range(1, n_epochs + 1):
        start = time.time()
        loader = make_train_loader(train_split, use_image, rng, neg_rate)
        loss = train_one_epoch(model, loader, optimizer, scaler)
        val_prob = predict(model, val_loader)
        ap = average_precision_score(y_val, val_prob)

        history.append({"epoch": epoch, "train_loss": loss, "val_ap": ap,
                        "minutes": (time.time() - start) / 60})
        print(f"epoch {epoch:2d} | loss {loss:.4f} | val AP {ap:.4f} | "
              f"{history[-1]['minutes']:.1f} min")

        if fixed_epochs:
            continue
        if ap > best_ap:
            best_ap, best_epoch, bad_epochs = ap, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"Early stopping. Best epoch: {best_epoch} (AP {best_ap:.4f})")
                break

    if not fixed_epochs:
        model.load_state_dict(best_state)

    return model, pd.DataFrame(history), best_epoch, best_ap

In [27]:
del model, optimizer, scaler, train_loader, val_loader   # free the speed-test objects
torch.cuda.empty_cache()

model_tab, history_tab, best_epoch_tab, best_ap_tab = fit(
    tune_train, tune_val, use_image=False, seed=0, neg_rate=0.25
)

epoch  1 | loss 0.4234 | val AP 0.2912 | 0.1 min
epoch  2 | loss 0.4089 | val AP 0.2993 | 0.1 min
epoch  3 | loss 0.4059 | val AP 0.3000 | 0.1 min
epoch  4 | loss 0.4049 | val AP 0.2995 | 0.1 min
epoch  5 | loss 0.4039 | val AP 0.2975 | 0.1 min
epoch  6 | loss 0.4037 | val AP 0.2991 | 0.1 min
Early stopping. Best epoch: 3 (AP 0.3000)


In [28]:
WORK_DIR = Path("/kaggle/working")

tuning_results = [{"model": "tabular", "neg_rate": 0.25,
                   "best_epoch": best_epoch_tab, "best_val_ap": best_ap_tab}]
history_tab.assign(model="tabular", neg_rate=0.25).to_csv(
    WORK_DIR / "history_tabular_r0.25.csv", index=False)

del model_tab
torch.cuda.empty_cache()

for use_image, neg_rate in [(False, 1.0), (True, 0.25), (True, 1.0)]:
    name = "image" if use_image else "tabular"
    print(f"\n=== {name}, r = {neg_rate} ===")

    model_, history_, best_epoch_, best_ap_ = fit(
        tune_train, tune_val, use_image=use_image, seed=0, neg_rate=neg_rate
    )

    history_.assign(model=name, neg_rate=neg_rate).to_csv(
        WORK_DIR / f"history_{name}_r{neg_rate}.csv", index=False)
    tuning_results.append({"model": name, "neg_rate": neg_rate,
                           "best_epoch": best_epoch_, "best_val_ap": best_ap_})

    del model_
    torch.cuda.empty_cache()

tuning_results = pd.DataFrame(tuning_results)
tuning_results.to_csv(WORK_DIR / "tuning_results.csv", index=False)
tuning_results


=== tabular, r = 1.0 ===
epoch  1 | loss 0.2019 | val AP 0.2957 | 0.2 min
epoch  2 | loss 0.1943 | val AP 0.2985 | 0.2 min
epoch  3 | loss 0.1934 | val AP 0.2945 | 0.2 min
epoch  4 | loss 0.1929 | val AP 0.2955 | 0.2 min
epoch  5 | loss 0.1924 | val AP 0.2904 | 0.2 min
Early stopping. Best epoch: 2 (AP 0.2985)

=== image, r = 0.25 ===
epoch  1 | loss 0.4095 | val AP 0.2960 | 1.4 min
epoch  2 | loss 0.3987 | val AP 0.3021 | 1.4 min
epoch  3 | loss 0.3948 | val AP 0.3027 | 1.4 min
epoch  4 | loss 0.3922 | val AP 0.3011 | 1.4 min
epoch  5 | loss 0.3904 | val AP 0.2994 | 1.4 min
epoch  6 | loss 0.3891 | val AP 0.3004 | 1.4 min
Early stopping. Best epoch: 3 (AP 0.3027)

=== image, r = 1.0 ===
epoch  1 | loss 0.1954 | val AP 0.3018 | 4.1 min
epoch  2 | loss 0.1910 | val AP 0.3025 | 4.1 min
epoch  3 | loss 0.1901 | val AP 0.2969 | 4.1 min
epoch  4 | loss 0.1890 | val AP 0.2971 | 4.1 min
epoch  5 | loss 0.1883 | val AP 0.2967 | 4.1 min
Early stopping. Best epoch: 2 (AP 0.3025)


,model,neg_rate,best_epoch,best_val_ap
0,tabular,0.25,3,0.299966
1,tabular,1.00,2,0.298520
2,image,0.25,3,0.302685
3,image,1.00,2,0.302487


### Tuning results (train 2017–2021, validation 2022, seed 0)

| Network | r = 0.25 | r = 1.0 | Best epoch |
|---|---:|---:|---:|
| Tabular only | 0.3000 | 0.2985 | 3 / 2 |
| Image + tabular | 0.3028 | 0.3024 | 3 / 2 |

**Negative sampling rate.** Both rates give practically identical results (differences of
0.0003–0.0015 in AP). The simpler option, **r = 1.0** (all observations, no probability
correction), is retained.

**Number of epochs.** Both networks reach their best validation AP after 2–3 epochs and
then degrade slightly, a sign of early overfitting. Final models are trained for
**2 epochs**, the best epoch with r = 1.0.

**Image contribution.** With a single seed, the image network is 0.003 AP above the
tabular network on 2022. Whether this difference is real is assessed in the final phase
with three seeds.

**Limitation.** Hyperparameter tuning of the image network was deliberately limited
(learning rates were not explored), to stay within a simple and reproducible protocol.

## 4. Final models (train 2017–2022, evaluation 2023)

Each network is trained for the 2 epochs selected in tuning, with r = 1.0, on the full
2017–2022 period, using three random seeds (0, 1, 2). Validation AP is printed after each
epoch for monitoring only; no training decision is based on 2023.

The predicted probabilities for 2023 are saved for every run, so that the networks can
be compared with each other and with the tabular models of Notebook 04 on exactly the
same observations.

In [29]:
FINAL_EPOCHS = 2
FINAL_NEG_RATE = 1.0
SEEDS = [0, 1, 2]

final_results = []
predictions = pd.DataFrame({
    "cell_id": final_val["cell_id"],
    "reference_date": final_val["reference_date"],
    "y": final_val["y"].astype(int),
})

for use_image in [False, True]:
    name = "image" if use_image else "tabular"
    for seed in SEEDS:
        print(f"\n=== {name}, seed {seed} ===")
        model_, history_, _, _ = fit(
            final_train, final_val, use_image=use_image, seed=seed,
            neg_rate=FINAL_NEG_RATE, fixed_epochs=FINAL_EPOCHS,
        )

        prob = predict(model_, make_eval_loader(final_val, use_image))
        predictions[f"{name}_seed{seed}"] = prob
        final_results.append({
            "model": name, "seed": seed,
            "val_ap_2023": average_precision_score(final_val["y"], prob),
        })

        torch.save(model_.state_dict(), WORK_DIR / f"model_{name}_seed{seed}.pt")
        del model_
        torch.cuda.empty_cache()

predictions.to_parquet(WORK_DIR / "cnn_predictions_2023.parquet", index=False)

final_results = pd.DataFrame(final_results)
final_results.to_csv(WORK_DIR / "final_results_2023.csv", index=False)

print(final_results)
print()
print(final_results.groupby("model")["val_ap_2023"].agg(["mean", "std", "min", "max"]))


=== tabular, seed 0 ===
epoch  1 | loss 0.1957 | val AP 0.2298 | 0.2 min
epoch  2 | loss 0.1890 | val AP 0.2308 | 0.2 min

=== tabular, seed 1 ===
epoch  1 | loss 0.1950 | val AP 0.2321 | 0.2 min
epoch  2 | loss 0.1888 | val AP 0.2338 | 0.2 min

=== tabular, seed 2 ===
epoch  1 | loss 0.1946 | val AP 0.2326 | 0.2 min
epoch  2 | loss 0.1886 | val AP 0.2329 | 0.2 min

=== image, seed 0 ===
epoch  1 | loss 0.1900 | val AP 0.2329 | 4.8 min
epoch  2 | loss 0.1861 | val AP 0.2377 | 4.8 min

=== image, seed 1 ===
epoch  1 | loss 0.1897 | val AP 0.2390 | 4.8 min
epoch  2 | loss 0.1861 | val AP 0.2400 | 4.8 min

=== image, seed 2 ===
epoch  1 | loss 0.1897 | val AP 0.2365 | 4.8 min
epoch  2 | loss 0.1858 | val AP 0.2367 | 4.8 min
     model  seed  val_ap_2023
0  tabular     0     0.230776
1  tabular     1     0.233772
2  tabular     2     0.232876
3    image     0     0.237654
4    image     1     0.240029
5    image     2     0.236712

             mean       std       min       max
model    

### Final results on 2023 (three seeds)

| Network | Mean AP | Std | Range |
|---|---:|---:|---|
| Tabular only | 0.2325 | 0.0015 | 0.2308–0.2338 |
| Image + tabular | 0.2381 | 0.0017 | 0.2367–0.2400 |

The image network is **0.0057 AP** above the tabular network, about 3.5 times the
variation across seeds, and every image run outperforms every tabular run. The gain is
therefore not explained by training randomness.

For reference, the best tabular model of Notebook 04 (XGB Weather + Context) reached
**AP = 0.2385** on the same year. The tabular network is weaker than XGBoost, which is
common for neural networks on tabular data; the image branch brings the network to the
same level. Whether the image adds information beyond what XGBoost already extracts is
assessed in Notebook 07.

In [31]:
def paired_bootstrap_ap(y_true, score_a, score_b, weeks, n_boot=1000, seed=42):
    """AP(score_b) - AP(score_a), resampling whole weeks with replacement."""
    rng = np.random.default_rng(seed)
    y_true, score_a, score_b, weeks = map(np.asarray, (y_true, score_a, score_b, weeks))
    unique_weeks = np.unique(weeks)
    week_rows = {w: np.flatnonzero(weeks == w) for w in unique_weeks}

    diffs = []
    for _ in range(n_boot):
        sampled = rng.choice(unique_weeks, size=len(unique_weeks), replace=True)
        idx = np.concatenate([week_rows[w] for w in sampled])
        diffs.append(average_precision_score(y_true[idx], score_b[idx])
                     - average_precision_score(y_true[idx], score_a[idx]))

    diffs = np.array(diffs)
    return {"mean_diff": diffs.mean(),
            "ci_low": np.percentile(diffs, 2.5),
            "ci_high": np.percentile(diffs, 97.5),
            "share_positive": (diffs > 0).mean()}


predictions["tabular_mean"] = predictions[[f"tabular_seed{s}" for s in SEEDS]].mean(axis=1)
predictions["image_mean"] = predictions[[f"image_seed{s}" for s in SEEDS]].mean(axis=1)
predictions.to_parquet(WORK_DIR / "cnn_predictions_2023.parquet", index=False)

for name in ["tabular_mean", "image_mean"]:
    print(f"{name}: AP = {average_precision_score(predictions['y'], predictions[name]):.4f}")

bootstrap_cnn = paired_bootstrap_ap(
    predictions["y"], predictions["tabular_mean"], predictions["image_mean"],
    predictions["reference_date"],
)
bootstrap_cnn

tabular_mean: AP = 0.2339
image_mean: AP = 0.2398


{'mean_diff': np.float64(0.005779884506744652),
 'ci_low': np.float64(0.002925904469184748),
 'ci_high': np.float64(0.009030794546705766),
 'share_positive': np.float64(1.0)}

### Final results (train 2017–2022, evaluation 2023)

| Network | Seed 0 | Seed 1 | Seed 2 | Mean ± std | 3-seed ensemble |
|---|---:|---:|---:|---:|---:|
| Tabular only | 0.2308 | 0.2338 | 0.2329 | 0.2325 ± 0.0015 | 0.2339 |
| Image + tabular | 0.2371 | 0.2402 | 0.2371 | 0.2381 ± 0.0018 | **0.2398** |

**Training variability.** The worst image-network seed (0.2371) outperforms the best
tabular-network seed (0.2338). The difference between the two networks is about 3.5 times
the standard deviation across seeds, so it is not an artefact of random initialisation.

**Sampling variability.** Using the average of the three seeds for each network, a paired
block bootstrap over weeks (1,000 resamples) gives:

| Comparison | Mean ΔAP | 95% CI | Share > 0 |
|---|---:|---|---:|
| Tabular network → Image + tabular network | +0.0058 | [+0.0029, +0.0090] | 100% |

**Conclusion of this notebook.** Within the same neural architecture, adding the previous
month's Sentinel-2 composite produces a small but robust improvement in Average Precision
(≈ +2.5% relative). This is the first satellite contribution in the project that is
distinguishable from noise.

**Caveat.** The tabular network (0.2339) is weaker than the best tree-based model of
Notebook 04, XGB Weather + Context (0.2385), which uses the same features. The image
network (0.2398) is practically level with it. Part of the image gain may therefore
compensate for the weaker tabular modelling of the network rather than add new
information. Whether the image adds information **beyond the best tabular model** is
tested in Notebook 07 by combining the predictions of both models.

In [20]:
WORK_DIR = Path("/kaggle/working")
SEEDS = [0, 1, 2]

weight_dirs = [
    Path(root) for root, _, files in os.walk("/kaggle/input")
    if "model_image_seed0.pt" in files
]
assert len(weight_dirs) == 1, weight_dirs
WEIGHTS_DIR = weight_dirs[0]

final_train, final_val = prepare_split(range(2017, 2023), 2023)

check_model = FireNet(len(TABULAR_FEATURES), use_image=True).to(DEVICE)
check_model.load_state_dict(
    torch.load(WEIGHTS_DIR / "model_image_seed0.pt", map_location=DEVICE)
)
check_prob = predict(check_model, make_eval_loader(final_val, use_image=True))
check_ap = average_precision_score(final_val["y"], check_prob)

print(f"Reloaded image seed 0, AP 2023: {check_ap:.4f} (expected 0.2371)")
assert abs(check_ap - 0.2371) < 0.002

del check_model
torch.cuda.empty_cache()

Reloaded image seed 0, AP 2023: 0.2377 (expected 0.2371)


## 5. Out-of-fold predictions for calibration

To calibrate the final combined model on several years rather than on 2023 alone, the
image network is retrained with the same settings (2 epochs, r = 1.0, three seeds) on
three expanding-window folds:

| Fold | Training | Predicted year |
|---|---|---|
| 1 | 2017–2019 | 2020 |
| 2 | 2017–2020 | 2021 |
| 3 | 2017–2021 | 2022 |

Each year is predicted by networks that never saw it. Together with the 2023 predictions
of the final networks, this gives four years of out-of-sample predictions (2020–2023)
for fitting the calibration in Notebook 07.

## 6. Test predictions (2024–2025)

The final networks (trained on 2017–2022) are applied to the test period with the same
preprocessing. **No metric is computed here**: the predictions are saved without labels
and evaluated once, in Notebook 07.

In [21]:
OOF_FOLDS = [
    (range(2017, 2020), 2020),
    (range(2017, 2021), 2021),
    (range(2017, 2022), 2022),
]

oof_parts = []

for train_years, val_year in OOF_FOLDS:
    fold_train, fold_val = prepare_split(train_years, val_year)

    fold_pred = pd.DataFrame({
        "cell_id": fold_val["cell_id"],
        "reference_date": fold_val["reference_date"],
        "y": fold_val["y"].astype(int),
    })

    for seed in SEEDS:
        print(f"\n=== fold → {val_year}, image, seed {seed} ===")
        model_, _, _, _ = fit(
            fold_train, fold_val, use_image=True, seed=seed,
            neg_rate=1.0, fixed_epochs=2,
        )
        fold_pred[f"image_seed{seed}"] = predict(
            model_, make_eval_loader(fold_val, use_image=True)
        )
        del model_
        torch.cuda.empty_cache()

    fold_pred["image_mean"] = fold_pred[[f"image_seed{s}" for s in SEEDS]].mean(axis=1)
    fold_pred.to_parquet(WORK_DIR / f"cnn_oof_{val_year}.parquet", index=False)
    oof_parts.append(fold_pred)

oof = pd.concat(oof_parts, ignore_index=True)
oof.to_parquet(WORK_DIR / "cnn_oof_2020_2022.parquet", index=False)
print(oof.groupby(oof["reference_date"].dt.year)["y"].agg(["size", "sum"]))


=== fold → 2020, image, seed 0 ===
epoch  1 | loss 0.2129 | val AP 0.3070 | 2.7 min
epoch  2 | loss 0.2071 | val AP 0.3035 | 2.6 min

=== fold → 2020, image, seed 1 ===
epoch  1 | loss 0.2128 | val AP 0.3098 | 2.6 min
epoch  2 | loss 0.2074 | val AP 0.3134 | 2.6 min

=== fold → 2020, image, seed 2 ===
epoch  1 | loss 0.2126 | val AP 0.3025 | 2.6 min
epoch  2 | loss 0.2071 | val AP 0.3134 | 2.6 min

=== fold → 2021, image, seed 0 ===
epoch  1 | loss 0.2057 | val AP 0.2266 | 3.3 min
epoch  2 | loss 0.2005 | val AP 0.2273 | 3.3 min

=== fold → 2021, image, seed 1 ===
epoch  1 | loss 0.2050 | val AP 0.2257 | 3.3 min
epoch  2 | loss 0.2006 | val AP 0.2258 | 3.3 min

=== fold → 2021, image, seed 2 ===
epoch  1 | loss 0.2048 | val AP 0.2248 | 3.3 min
epoch  2 | loss 0.2002 | val AP 0.2269 | 3.3 min

=== fold → 2022, image, seed 0 ===
epoch  1 | loss 0.1954 | val AP 0.3005 | 4.0 min
epoch  2 | loss 0.1910 | val AP 0.3012 | 4.1 min

=== fold → 2022, image, seed 1 ===
epoch  1 | loss 0.1953 | v

In [22]:
_, test_split = prepare_split(range(2017, 2023), [2024, 2025])

# Labels are deliberately NOT saved: evaluation happens only in Notebook 07
test_pred = pd.DataFrame({
    "cell_id": test_split["cell_id"],
    "reference_date": test_split["reference_date"],
})

for name, use_image in [("tabular", False), ("image", True)]:
    for seed in SEEDS:
        model_ = FireNet(len(TABULAR_FEATURES), use_image=use_image).to(DEVICE)
        model_.load_state_dict(
            torch.load(WEIGHTS_DIR / f"model_{name}_seed{seed}.pt", map_location=DEVICE)
        )
        test_pred[f"{name}_seed{seed}"] = predict(
            model_, make_eval_loader(test_split, use_image=use_image)
        )
        del model_
        torch.cuda.empty_cache()
        print(f"{name} seed {seed}: done")

test_pred["tabular_mean"] = test_pred[[f"tabular_seed{s}" for s in SEEDS]].mean(axis=1)
test_pred["image_mean"] = test_pred[[f"image_seed{s}" for s in SEEDS]].mean(axis=1)

test_pred.to_parquet(WORK_DIR / "cnn_predictions_test_2024_2025.parquet", index=False)
print(test_pred.shape)

tabular seed 0: done
tabular seed 1: done
tabular seed 2: done
image seed 0: done
image seed 1: done
image seed 2: done
(191100, 10)
